In [24]:
!git clone https://github.com/tnisl/PytorchNeuralStyleTransfer.git

Cloning into 'PytorchNeuralStyleTransfer'...
remote: Enumerating objects: 33, done.
remote: Counting objects: 100% (6/6), done.
remote: Compressing objects: 100% (5/5), done.
remote: Total 33 (delta 0), reused 5 (delta 0), pack-reused 27 (from 1)
Receiving objects: 100% (33/33), 2.47 MiB | 46.92 MiB/s, done.
Resolving deltas: 100% (9/9), done.


In [25]:
%cd /kaggle/working/PytorchNeuralStyleTransfer

/kaggle/working/PytorchNeuralStyleTransfer


In [26]:
pip install matplotlib torch torchvision

Note: you may need to restart the kernel to use updated packages.


In [27]:
%%bash
set -euo pipefail

mkdir -p Models

pip install gdown
python -m gdown \
  "https://drive.google.com/uc?id=1lLSi8BXd_9EtudRbIwxvmTQ3Ms-Qh6C8" \
  -O Models/vgg_conv.pth

ls -lh Models/vgg_conv.pth

-rw-r--r-- 1 root root 153M Dec 20  2021 Models/vgg_conv.pth


Downloading...
From (original): https://drive.google.com/uc?id=1lLSi8BXd_9EtudRbIwxvmTQ3Ms-Qh6C8
From (redirected): https://drive.google.com/uc?id=1lLSi8BXd_9EtudRbIwxvmTQ3Ms-Qh6C8&confirm=t&uuid=1eb156f2-bf15-4640-a02e-7fc13d60cd74
To: /kaggle/working/PytorchNeuralStyleTransfer/Models/vgg_conv.pth
100%|██████████| 160M/160M [00:02<00:00, 57.9MB/s] 


In [28]:
%pylab inline
import time
import os 
image_dir = os.getcwd() + '/Images/'
model_dir = os.getcwd() + '/Models/'

import torch
from torch.autograd import Variable
import torch.nn as nn
import torch.nn.functional as F
from torch import optim

import torchvision
from torchvision import transforms

from PIL import Image
from collections import OrderedDict


Populating the interactive namespace from numpy and matplotlib


In [29]:
#vgg definition that conveniently let's you grab the outputs from any layer
class VGG(nn.Module):
    def __init__(self, pool='max'):
        super(VGG, self).__init__()
        #vgg modules
        self.conv1_1 = nn.Conv2d(3, 64, kernel_size=3, padding=1)
        self.conv1_2 = nn.Conv2d(64, 64, kernel_size=3, padding=1)
        self.conv2_1 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.conv2_2 = nn.Conv2d(128, 128, kernel_size=3, padding=1)
        self.conv3_1 = nn.Conv2d(128, 256, kernel_size=3, padding=1)
        self.conv3_2 = nn.Conv2d(256, 256, kernel_size=3, padding=1)
        self.conv3_3 = nn.Conv2d(256, 256, kernel_size=3, padding=1)
        self.conv3_4 = nn.Conv2d(256, 256, kernel_size=3, padding=1)
        self.conv4_1 = nn.Conv2d(256, 512, kernel_size=3, padding=1)
        self.conv4_2 = nn.Conv2d(512, 512, kernel_size=3, padding=1)
        self.conv4_3 = nn.Conv2d(512, 512, kernel_size=3, padding=1)
        self.conv4_4 = nn.Conv2d(512, 512, kernel_size=3, padding=1)
        self.conv5_1 = nn.Conv2d(512, 512, kernel_size=3, padding=1)
        self.conv5_2 = nn.Conv2d(512, 512, kernel_size=3, padding=1)
        self.conv5_3 = nn.Conv2d(512, 512, kernel_size=3, padding=1)
        self.conv5_4 = nn.Conv2d(512, 512, kernel_size=3, padding=1)
        if pool == 'max':
            self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)
            self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)
            self.pool3 = nn.MaxPool2d(kernel_size=2, stride=2)
            self.pool4 = nn.MaxPool2d(kernel_size=2, stride=2)
            self.pool5 = nn.MaxPool2d(kernel_size=2, stride=2)
        elif pool == 'avg':
            self.pool1 = nn.AvgPool2d(kernel_size=2, stride=2)
            self.pool2 = nn.AvgPool2d(kernel_size=2, stride=2)
            self.pool3 = nn.AvgPool2d(kernel_size=2, stride=2)
            self.pool4 = nn.AvgPool2d(kernel_size=2, stride=2)
            self.pool5 = nn.AvgPool2d(kernel_size=2, stride=2)
            
    def forward(self, x, out_keys):
        out = {}
        out['r11'] = F.relu(self.conv1_1(x))
        out['r12'] = F.relu(self.conv1_2(out['r11']))
        out['p1'] = self.pool1(out['r12'])
        out['r21'] = F.relu(self.conv2_1(out['p1']))
        out['r22'] = F.relu(self.conv2_2(out['r21']))
        out['p2'] = self.pool2(out['r22'])
        out['r31'] = F.relu(self.conv3_1(out['p2']))
        out['r32'] = F.relu(self.conv3_2(out['r31']))
        out['r33'] = F.relu(self.conv3_3(out['r32']))
        out['r34'] = F.relu(self.conv3_4(out['r33']))
        out['p3'] = self.pool3(out['r34'])
        out['r41'] = F.relu(self.conv4_1(out['p3']))
        out['r42'] = F.relu(self.conv4_2(out['r41']))
        out['r43'] = F.relu(self.conv4_3(out['r42']))
        out['r44'] = F.relu(self.conv4_4(out['r43']))
        out['p4'] = self.pool4(out['r44'])
        out['r51'] = F.relu(self.conv5_1(out['p4']))
        out['r52'] = F.relu(self.conv5_2(out['r51']))
        out['r53'] = F.relu(self.conv5_3(out['r52']))
        out['r54'] = F.relu(self.conv5_4(out['r53']))
        out['p5'] = self.pool5(out['r54'])
        return [out[key] for key in out_keys]

In [30]:
# gram matrix and loss
class GramMatrix(nn.Module):
    def forward(self, input):
        b,c,h,w = input.size()
        F = input.view(b, c, h*w)
        G = torch.bmm(F, F.transpose(1,2)) 
        G.div_(h*w)
        return G

class GramMSELoss(nn.Module):
    def forward(self, input, target):
        out = nn.MSELoss()(GramMatrix()(input), target)
        return(out)

In [31]:
# pre and post processing for images
img_size = 512 
prep = transforms.Compose([transforms.Resize(img_size),
                           transforms.ToTensor(),
                           transforms.Lambda(lambda x: x[torch.LongTensor([2,1,0])]), #turn to BGR
                           transforms.Normalize(mean=[0.40760392, 0.45795686, 0.48501961], #subtract imagenet mean
                                                std=[1,1,1]),
                           transforms.Lambda(lambda x: x.mul_(255)),
                          ])
postpa = transforms.Compose([transforms.Lambda(lambda x: x.mul_(1./255)),
                           transforms.Normalize(mean=[-0.40760392, -0.45795686, -0.48501961], #add imagenet mean
                                                std=[1,1,1]),
                           transforms.Lambda(lambda x: x[torch.LongTensor([2,1,0])]), #turn to RGB
                           ])
postpb = transforms.Compose([transforms.ToPILImage()])
def postp(tensor): # to clip results in the range [0,1]
    t = postpa(tensor)
    t[t>1] = 1    
    t[t<0] = 0
    img = postpb(t)
    return img

In [32]:
#get network
vgg = VGG()
vgg.load_state_dict(torch.load(model_dir + 'vgg_conv.pth'))
for param in vgg.parameters():
    param.requires_grad = False
if torch.cuda.is_available():
    vgg.cuda()

In [33]:
#load images, ordered as [style_image, content_image]
img_dirs = [image_dir, image_dir]
img_names = ['style.jpg', 'content.jpg']
imgs = [Image.open(img_dirs[i] + name) for i,name in enumerate(img_names)]
imgs_torch = [prep(img) for img in imgs]
if torch.cuda.is_available():
    imgs_torch = [Variable(img.unsqueeze(0).cuda()) for img in imgs_torch]
else:
    imgs_torch = [Variable(img.unsqueeze(0)) for img in imgs_torch]
style_image, content_image = imgs_torch

opt_img = Variable(torch.randn(content_image.size()).type_as(content_image.data) * (1e-3), requires_grad=True) #random init
# opt_img = Variable(content_image.data.clone(), requires_grad=True)

In [35]:
#define layers, loss functions, weights and compute optimization targets
style_layers = ['r11','r21','r31','r41', 'r51'] 
content_layers = ['r42']
loss_layers = style_layers + content_layers
loss_fns = [GramMSELoss()] * len(style_layers) + [nn.MSELoss()] * len(content_layers)
if torch.cuda.is_available():
    loss_fns = [loss_fn.cuda() for loss_fn in loss_fns]
    
#these are good weights settings:
style_weights = [1e3/n**2 for n in [64,128,256,512,512]]
content_weights = [1e0]
weights = style_weights + content_weights

#compute optimization targets
style_targets = [GramMatrix()(A).detach() for A in vgg(style_image, style_layers)]
content_targets = [A.detach() for A in vgg(content_image, content_layers)]
targets = style_targets + content_targets

In [ ]:
device = opt_img.device

weights = [float(w) for w in weights]
targets = [t.to(device) for t in targets]

max_iter = 50000
show_iter = 50

optimizer = optim.LBFGS([opt_img])
n_iter = [0]

while n_iter[0] <= max_iter:

    def closure():
        optimizer.zero_grad()

        out = vgg(opt_img, loss_layers)

        layer_losses = [
            weights[a] * loss_fns[a](activation, targets[a])
            for a, activation in enumerate(out)
        ]

        loss = torch.stack(layer_losses).sum()
        loss.backward()

        n_iter[0] += 1

        if n_iter[0] % show_iter == show_iter - 1:
            print(f"Iteration: {n_iter[0] + 1}, loss: {loss.item():.6f}")

        return loss

    optimizer.step(closure)

# display result
out_img = postp(opt_img.detach()[0].cpu().squeeze())
imshow(out_img)
gcf().set_size_inches(10, 10)

Iteration: 50, loss: 15125158.000000
Iteration: 100, loss: 2729211.000000
Iteration: 150, loss: 1432822.625000
Iteration: 200, loss: 1057708.000000
Iteration: 250, loss: 885491.750000
Iteration: 300, loss: 784141.250000
Iteration: 350, loss: 717443.125000
Iteration: 400, loss: 666548.937500
Iteration: 450, loss: 626733.562500
Iteration: 500, loss: 596213.187500
Iteration: 550, loss: 571174.500000
Iteration: 600, loss: 552212.312500
Iteration: 650, loss: 536022.437500
Iteration: 700, loss: 523152.562500
Iteration: 750, loss: 512071.937500
Iteration: 800, loss: 502500.593750
Iteration: 850, loss: 494302.125000
Iteration: 900, loss: 487099.031250
Iteration: 950, loss: 480800.187500
Iteration: 1000, loss: 475210.500000
Iteration: 1050, loss: 470321.906250
Iteration: 1100, loss: 465681.781250
Iteration: 1150, loss: 461378.781250
Iteration: 1200, loss: 457457.531250
Iteration: 1250, loss: 453829.656250
Iteration: 1300, loss: 450609.562500
Iteration: 1350, loss: 447460.718750
Iteration: 1400,